# Libraries

In [ ]:
import os
import numpy as np
import zarr
import torch
import napari
from pathlib import Path

# Device initialization

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Visualization

In [ ]:
"""ArithmeticError# Load your sample zarr path
train_dir = Path("/home/upmanyu/ml-workspace/Biohub/data/biohub-cell-tracking-during-development/train")
sample_name = [f for f in os.listdir(train_dir) if f.endswith(".zarr")][10]
sample_path = os.path.join(train_dir, sample_name)

# Open zarr container
zarr_group = zarr.open_group(sample_path, mode="r")
image_array = zarr_group["0"]  # Shape: (T, Z, Y, X)

viewer = napari.Viewer()
viewer.add_image(image_array, name="Cell Tracking volume",
                 colormap='inferno',
                 scale=(1.625, 0.40625, 0.40625))
napari.run()
"""

# Data Reader Functions

In [ ]:
import json
def read_array_meta(dataset_zarr_path: Path):

    dataset_zarr_path = Path(dataset_zarr_path) / "0" / "zarr.json"

    with open(dataset_zarr_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    return meta

def estimated_cells(dataset_geff_path: Path):
    dataset_geff_path = Path(dataset_geff_path)/ "0" / "zarr.json"

    with open(dataset_geff_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    return meta["extra"]["estimated_number_of_nodes"]

def read_geff_nodes(geff_path):

    geff_path = Path(geff_path)
    label_group = zarr.open_group(str(geff_path), mode="r")
    
    node_ids = label_group["nodes/ids"][:]
    nodes_t = label_group["nodes/props/t/values"][:]
    nodes_z = label_group["nodes/props/z/values"][:]
    nodes_y = label_group["nodes/props/y/values"][:]
    nodes_x = label_group["nodes/props/x/values"][:]
    
    return {
        "ids": node_ids,
        "t": nodes_t,
        "z": nodes_z,
        "y": nodes_y,
        "x": nodes_x
    }

def load_volume(zarr_path, t_idx=None):

    zarr_path = Path(zarr_path)
    img_group = zarr.open_group(str(zarr_path), mode="r")
    img_array = img_group["0"]  # Primary image array path

    vol = img_array[t_idx]

    return vol

# Normalization function

In [ ]:
def normalize(vol):
    """
    Normalizes a 3D microscopy volume using 2nd and 98th percentiles 
    to handle outliers, then scales the values to the range [0, 1].
    """
    vol = vol.astype(np.float32)
    
    # Calculate 2nd and 98th percentiles to avoid extreme intensity outliers
    p_low, p_high = np.percentile(vol, (2, 98))
    
    # Clip values outside this range
    vol = np.clip(vol, p_low, p_high)
    
    # Scale safely to [0, 1] (preventing division by zero if the slice is completely blank)
    if p_high - p_low > 1e-5:
        vol = (vol - p_low) / (p_high - p_low)
    else:
        vol = np.zeros_like(vol)
        
    return vol

# HeatMap Generator

In [ ]:
def heatmap_generator(shape, pts, cell_radius_um=5, dz=1.625, dy=0.40625, dx=0.40625):
    """
    Generates a 3D target heatmap containing multiple cell centers for a single frame.
    
    Args:
        shape (tuple): (Z, Y, X) dimensions of the volume patch or full frame.
        pts (np.ndarray): Array of shape (N, 3) containing [z, y, x] coordinates of all cells in frame `t`.
        cell_radius_um (float): Estimated physical radius of the cells.
        dz, dy, dx (float): Physical voxel spacing in micrometers.
        
    Returns:
        np.ndarray: 3D float32 array of shape (Z, Y, X) with values ranging from 0.0 to 1.0.
    """
    # 1. Initialize an empty canvas for the frame
    canvas = np.zeros(shape, dtype=np.float32)
    
    if len(pts) == 0:
        return canvas  # Return blank canvas if no cells are present in this frame
        
    # 2. Create global coordinate grid once for efficiency
    Z, Y, X = shape
    zz, yy, xx = np.ogrid[:Z, :Y, :X]
    
    sigma_physical = cell_radius_um / 2.0
    """debug_deno = []
    debug_gauss = []
    debug_num = []"""
    
    # 3. Loop through every cell node in this frame and paint its Gaussian
    for pt in pts:
        cz, cy, cx = pt
        
        # Using Gaussian formula for heatmap generation
        dist_sq = (((zz - cz) * dz) ** 2) / (2 * sigma_physical ** 2) + \
                  (((yy - cy) * dy) ** 2) / (2 * sigma_physical ** 2) + \
                  (((xx - cx) * dx) ** 2) / (2 * sigma_physical ** 2)
        
        # Generate the Gaussian blob for this cell
        gaussian = np.exp(-dist_sq)
        
        # Merge onto the canvas using max (prevents value clipping/overflow on overlap)
        canvas = np.maximum(canvas, gaussian)
        
    return canvas  

# Weight Map Generator

In [ ]:
def weight_map_gen(target_vol, input_vol, bg_wt = 0.0 ,low_wt = 0.1 ,bg_thesh = 0.30 ,cell_thresh = 0.1 ,cell_wt = 5):
    
    # 1. Initialize entire patch with the neutral/ignore weight
    weight_map = np.full(target_vol.shape, bg_wt, dtype=np.float32)
    
    # 2. Confident background: low optical intensity
    bg_cutoff = float(np.quantile(input_vol, bg_thesh))
    weight_map[input_vol < bg_cutoff] = low_wt
    
    # 3. Confident positives: annotated Gaussian centers override background
    weight_map[target_vol > cell_thresh] = cell_wt
    
    return weight_map

# DataSet Creation

In [ ]:
from torch.utils.data import Dataset
class CellDataset(Dataset):

    def __init__(
        self,
        data_dir,
        fpm=20,
        augment=False
    ):
        self.data_dir = Path(data_dir)
        self.fpm = fpm
        self.augment = augment
        self.samples = []
        self._build_index()
        self.samples.sort()

    def __len__(self):
        return len(self.samples)
    
    def _build_index(self):
        zarr_files = sorted(self.data_dir.glob("*.zarr"))

        for z_path in zarr_files:
            ds_name = z_path.stem
            g_path = self.data_dir / f"{ds_name}.geff"

            # Require matching GEFF file
            if not g_path.exists():
                continue

            try:
                # Only read metadata, not the volume itself
                meta = read_array_meta(str(z_path))

                # Read node table
                nodes = read_geff_nodes(str(g_path))

                # Frames actually containing GEFF annotations
                node_t = nodes["t"]

                available_times = sorted(
                    set(node_t.tolist()) &
                    set(range(meta["shape"][0]))
                )

                if not available_times:
                    continue

                # Select fpm frames uniformly
                indices = np.linspace(
                    0,
                    len(available_times) - 1,
                    min(self.fpm, len(available_times))
                )

                for idx in indices:
                    t = available_times[idx]
                    self.samples.append((ds_name, t))

            except Exception as e:
                print(
                    f"Skipping {ds_name} due to read error: {e}"
                )

        # ! i had added sample sorting i guess it is repeated

        print(
            f"[Dataset] Indexed {len(self.samples)} "
            f"paired frames across all movies."
        )

    def __getitem__(self, idx):

        ds_name, t = self.samples[idx]

        z_path = str(self.data_dir / f"{ds_name}.zarr")
        g_path = str(self.data_dir / f"{ds_name}.geff")

        vol = load_volume(z_path, t)
        x = normalize(vol).astype(np.float32)

        nodes = read_geff_nodes(g_path)

        mask = nodes["t"] == t

        pts = np.stack(
            [
                nodes["z"][mask],
                nodes["y"][mask],
                nodes["x"][mask]
            ],
            axis=-1
        ).astype(np.float32)

        target_heatmap = heatmap_generator(
            x.shape,
            pts
        )

        weight_map = weight_map_gen(
            target_heatmap,
            x
        )

        return (
            torch.from_numpy(x).unsqueeze(0),
            torch.from_numpy(target_heatmap).unsqueeze(0),
            torch.from_numpy(weight_map).unsqueeze(0),
            torch.from_numpy(pts)
        )

# Model Architecture

In [ ]:
import torch
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_channels, mid_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels, mid_channels, kernel_size=3, padding='same', stride=1)
        self.batch1 = nn.BatchNorm3d(mid_channels)
        self.activation1 = nn.ReLU() 
        
        self.conv2 = nn.Conv3d(mid_channels, out_channels, kernel_size=3, padding='same', stride=1)
        self.batch2 = nn.BatchNorm3d(out_channels)
        self.activation2 = nn.ReLU()
        
    def forward(self, X):
        out1 = self.activation1(self.batch1(self.conv1(X)))
        out2 = self.activation2(self.batch2(self.conv2(out1)))
        return out2

class UNet3D(nn.Module):
    def __init__(self):
        super().__init__()
        
        # 1. Entry Stem (Your idea: learned downsampling to save VRAM)
        self.stem = nn.Sequential(
            nn.Conv3d(1, 32, kernel_size=3, stride=2, padding=1), # Halves spatial & depth size
            nn.BatchNorm3d(32),
            nn.ReLU()
        )
        
        # 2. Encoder Path (operates safely at half resolution)
        self.enc1 = ConvBlock(32, 32, 64)
        self.pool1 = nn.MaxPool3d(kernel_size=2, stride=2)
        self.enc2 = ConvBlock(64, 64, 128) # Bottleneck
        
        # 3. Decoder Path
        self.TransConv1 = nn.ConvTranspose3d(128, 128, kernel_size=2, stride=2)
        self.decod1 = ConvBlock(192, 64, 64) # 128 (from decoder) + 64 (from skip l1) = 192
        
        # 4. Final Upsample (Restores output back to full original size)
        self.final_upsample = nn.ConvTranspose3d(64, 64, kernel_size=2, stride=2)
        self.finalconv = nn.Conv3d(64, 1, kernel_size=1, padding='same', stride=1)
        
    def forward(self, X):
        # Input: (B, 1, 64, 256, 256)
        
        # Step 1: Entry Stem Compression
        stem_out = self.stem(X)          # Shape: (B, 32, 32, 128, 128)
        
        # Step 2: Encoder
        l1 = self.enc1(stem_out)         # Shape: (B, 64, 32, 128, 128) -> Saved for skip connection
        m1 = self.pool1(l1)              # Shape: (B, 64, 16, 64, 64)
        l2 = self.enc2(m1)               # Shape: (B, 128, 16, 64, 64) -> Bottleneck feature map
        
        # Step 3: Decoder & Skip Connection Integration
        t1 = self.TransConv1(l2)         # Shape: (B, 128, 32, 128, 128)
        c1 = torch.cat((l1, t1), dim=1)  # Shape: (B, 192, 32, 128, 128) -> Combined features!
        d1 = self.decod1(c1)             # Shape: (B, 64, 32, 128, 128)
        
        # Step 4: Restore Original Resolution
        restored = self.final_upsample(d1) # Shape: (B, 64, 64, 256, 256)
        out = self.finalconv(restored)     # Shape: (B, 1, 64, 256, 256)
        
        return out

# Loss Function

In [ ]:
from torch.nn.functional import binary_cross_entropy_with_logits as BCE
def weighted_loss(pred_vol, target_vol, weight_map):
    loss = BCE(pred_vol, target_vol, reduction="none")
    weighted = loss * weight_map
    return weighted.sum() / torch.clamp(weight_map.sum(), min=1.0) # to reduice the impace of the weight multiplication which increses the loss value

# Test metric

In [ ]:
from scipy.ndimage import maximum_filter
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
def hungarian_matching(pred_heatmap,nodes,box,intst_thresh):
    peaks_map = (
        (pred_heatmap == maximum_filter(pred_heatmap,size = box))
        & (pred_heatmap >= intst_thresh)
    )

    vox_to_um = [1.625, 0.40625, 0.40625]

    z, y, x = np.where(peaks_map)
    centers = np.stack([z, y, x], axis=-1).astype(np.float32)

    # converting voxel to um
    centers_um = centers * vox_to_um
    targets_um = nodes * vox_to_um

    # distance matrix
    distance_matrix = cdist(centers_um, targets_um, metric = "euclidean")

    # hungarian algorithm for bipartite graph
    center_idx , target_idx = linear_sum_assignment(distance_matrix)
    dist_matched = distance_matrix[center_idx, target_idx]

    return dist_matched


# Training Loop

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from tqdm import tqdm

def train_and_validate(
    model: nn.Module,
    train_loader: torch.utils.data.DataLoader,
    val_loader: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
    num_epochs: int,
    device: torch.device,
    recall_thresholds=(1.0, 2.0, 3.0, 5.0, 7.0)
):
    """
    Executes training and validation, records history, and plots learning curves.
    """
    model.to(device)
    
    # Track losses across epochs for diagnostics
    train_losses = []
    val_losses = []
    
    for epoch in range(num_epochs):
        # ==========================================
        # 1. TRAINING PHASE
        # ==========================================
        model.train()
        train_loss = 0.0
        
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for input_vol, target_vol, weight_map, _ in train_bar:
            input_vol = input_vol.to(device)
            target_vol = target_vol.to(device)
            weight_map = weight_map.to(device)
            
            optimizer.zero_grad()
            pred_vol = model(input_vol)
            loss = weighted_loss(pred_vol, target_vol, weight_map)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            train_bar.set_postfix(loss=f"{loss.item():.4f}")
            
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # ==========================================
        # 2. VALIDATION PHASE
        # ==========================================
        model.eval()
        val_loss = 0.0
        total_gt = 0
        all_matched_distances = []

        with torch.no_grad():

            val_bar = tqdm(
                val_loader,
                desc=f"Epoch {epoch+1}/{num_epochs} [Validation]"
            )

            for input_vol, target_vol, weight_map, nodes in val_bar:

                input_vol = input_vol.to(device)
                target_vol = target_vol.to(device)
                weight_map = weight_map.to(device)

                pred_vol = model(input_vol)

                # -------------------------
                # Validation loss
                # -------------------------
                loss = weighted_loss(
                    pred_vol,
                    target_vol,
                    weight_map
                )

                val_loss += loss.item()

                # -------------------------
                # Convert prediction
                # -------------------------
                pred_heatmap = (
                    pred_vol[0, 0]
                    .detach()
                    .cpu()
                    .numpy()
                )

                # -------------------------
                # Convert GT nodes
                # -------------------------
                gt_nodes = nodes[0].cpu().numpy()

                # -------------------------
                # Hungarian matching
                # -------------------------
                distance = hungarian_matching(
                    pred_heatmap,
                    gt_nodes,
                    box=7,
                    intst_thresh=0.6
                )

                all_matched_distances.extend(
                    distance.tolist()
                )

                total_gt += len(gt_nodes)

        # & Avg loss calculated
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        # & Recall for various distace thresholds
        all_matched_distances = np.asarray(all_matched_distances, dtype=np.float32)
        recall = {}
        for threshold in recall_thresholds:
            matched = np.sum(
                all_matched_distances <= threshold
            )
    
            recall[f"recall_{threshold:g}um"] = (
                matched / total_gt
                if total_gt > 0
                else 0.0
            )

        # & Calculatiing the distance for the center (optional)
        localization_distances = all_matched_distances[
            all_matched_distances < 7.0
        ]
    
        if len(localization_distances) > 0:
            mse = np.mean(localization_distances ** 2)
            rmse = np.sqrt(mse)
            p90 = np.percentile(localization_distances, 90)
            mean_distance = np.mean(localization_distances)
            median_distance = np.median(localization_distances)
        else:
            mse = np.nan
            rmse = np.nan
            p90 = np.nan
            mean_distance = np.nan
            median_distance = np.nan

        print(f"--- Epoch {epoch+1} Summary ---")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}\n")
        print("Recall:")
        for threshold, value in recall.items():
            print(f"  {threshold}: {value:.4f}")

        print("Localization:")
        print(f"  Mean:   {mean_distance:.4f} µm")
        print(f"  Median: {median_distance:.4f} µm")
        print(f"  RMSE:   {rmse:.4f} µm")
        print(f"  P90:    {p90:.4f} µm")

    # Automatically plot diagnostic curves once training completes
    plot_diagnostic_curves(train_losses, val_losses)
    
    return train_losses, val_losses

# Learning Curves

In [ ]:
def plot_diagnostic_curves(train_losses, val_losses):
    """
    Plots the Bias-Variance Diagnosis Curve and Convergence Diagnosis Curve.
    """
    epochs = range(1, len(train_losses) + 1)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # ------------------------------------------
    # Plot 1: Bias-Variance Diagnosis Curve
    # ------------------------------------------
    axes[0].plot(epochs, train_losses, label='Training Loss', color='blue', linewidth=2)
    axes[0].plot(epochs, val_losses, label='Validation Loss', color='orange', linewidth=2)
    axes[0].set_title('Bias-Variance Diagnosis Curve')
    axes[0].set_xlabel('Epochs')
    axes[0].set_ylabel('Weighted MSE Loss')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.6)
    
    # ------------------------------------------
    # Plot 2: Convergence Diagnosis Curve (Log Scale)
    # ------------------------------------------
    axes[1].plot(epochs, train_losses, label='Train Loss (Log)', color='blue', linestyle='--')
    axes[1].plot(epochs, val_losses, label='Val Loss (Log)', color='orange', linestyle='--')
    axes[1].set_yscale('log')
    axes[1].set_title('Convergence Diagnosis Curve (Log Scale)')
    axes[1].set_xlabel('Epochs')
    axes[1].set_ylabel('Loss (Log Scale)')
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    plt.show()

# Main function

In [ ]:
from torch.utils.data import Subset
from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader
from torch import optim
dataset = CellDataset(
    data_dir= "/home/upmanyu/ml-workspace/Biohub/data/biohub-cell-tracking-during-development/train",
    fpm= 5, # ! adjust for lower traning time
    augment= False
)
groups = np.array([movie for movie, frame in dataset.samples])
gkf = GroupKFold(n_splits=5, random_state = 47, shuffle = True)
for fold, (train_idx, val_idx) in enumerate(gkf.split(dataset, groups=groups)):
    print(f"--- Training on Fold {fold+1}/5 ---")
    
    train_dataset = Subset(dataset, train_idx)
    val_dataset = Subset(dataset, val_idx)
    
    # Re-initialize dataloaders and model for this fold
    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=10)
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=10)
    
    model = UNet3D().to(device)
    optimizer = optim.SGD(
    model.parameters(), 
    lr=1e-2  # Note: SGD typically requires a higher learning rate than AdamW (e.g., 1e-2 or 1e-1)
)
    
    # Run training for this fold
    train_and_validate(model, train_loader, val_loader, optimizer, num_epochs=5, device=device)
    break # ! added just to run one loop

# Model Saving

In [ ]:
from pathlib import Path
import torch

# Create a 'weights' directory in your project workspace if it doesn't exist
weights_dir = Path("weights")
weights_dir.mkdir(parents=True, exist_ok=True)

# Define the file path for your saved parameters
model_path = weights_dir / "unet3d_v1_exp1.pth"

# Save only the model's learned parameters (state_dict)
torch.save(model.state_dict(), model_path)

print(f"Model parameters successfully saved to: {model_path}")

# Tuning the bias-variance

# Experiment 1: Bias-Variance Curve

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

this a different type of data set where we will get the movies according to loop

In [ ]:
from torch.utils.data import Dataset

class CellDataset(Dataset):

    def __init__(
        self,
        data_dir,
        movies,
        fpm=20
    ):
        self.data_dir = Path(data_dir)
        self.fpm = fpm
        self.samples = []
        self.movies = movies

        self._build_index()
        self.samples.sort()

    def __len__(self):
        return len(self.samples)

    def _build_index(self):

        if self.movies is None:
            print("Got no files ----------------------------------")
            zarr_files = sorted(self.data_dir.glob("*.zarr"))
        else:
            zarr_files = sorted(self.movies)

        for z_path in zarr_files:

            # Get movie name from the zarr filename
            ds_name = z_path.stem

            # Corresponding GEFF file
            g_path = self.data_dir / f"{ds_name}.geff"

            # Require matching GEFF file
            if not g_path.exists():
                continue

            try:
                # Only read metadata, not the volume itself
                meta = read_array_meta(str(z_path))

                # Read node table
                nodes = read_geff_nodes(str(g_path))

                # Frames actually containing GEFF annotations
                node_t = nodes["t"]

                available_times = sorted(
                    set(node_t.tolist()) &
                    set(range(meta["shape"][0]))
                )

                if not available_times:
                    continue

                # Select fpm frames uniformly
                indices = np.linspace(
                    0,
                    len(available_times) - 1,
                    min(self.fpm, len(available_times))
                ).round().astype(int)

                for idx in indices:
                    t = available_times[idx]
                    self.samples.append((ds_name, t))

            except Exception as e:
                print(
                    f"Skipping {ds_name} due to read error: {e}"
                )

        print(
            f"[Dataset] Indexed {len(self.samples)} "
            f"paired frames across {len(zarr_files)} movies."
        )

    def __getitem__(self, idx):

        ds_name, t = self.samples[idx]

        z_path = str(self.data_dir / f"{ds_name}.zarr")
        g_path = str(self.data_dir / f"{ds_name}.geff")

        vol = load_volume(z_path, t)
        x = normalize(vol).astype(np.float32)

        nodes = read_geff_nodes(g_path)

        mask = nodes["t"] == t

        pts = np.stack(
            [
                nodes["z"][mask],
                nodes["y"][mask],
                nodes["x"][mask]
            ],
            axis=-1
        ).astype(np.float32)

        target_heatmap = heatmap_generator(
            x.shape,
            pts
        )

        weight_map = weight_map_gen(
            target_heatmap,
            x
        )

        return (
            torch.from_numpy(x).unsqueeze(0),
            torch.from_numpy(target_heatmap).unsqueeze(0),
            torch.from_numpy(weight_map).unsqueeze(0)
        )

In [ ]:
import copy
import numpy as np
import torch
from tqdm.auto import tqdm

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """
    Trains the model for one epoch.
    Collects individual batch losses, computes their mean, and returns it.
    """
    model.train()
    batch_losses = []

    pbar = tqdm(dataloader, desc="Training Batch", leave=False)
    
    # FIX 1: Unpack 3 items (input, target, weight_map)
    for X_batch, y_batch, w_batch in pbar:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        w_batch = w_batch.to(device)

        optimizer.zero_grad()
        y_pred = model(X_batch)
        
        # FIX 2: Pass all 3 arguments to the weighted_loss criterion
        loss = criterion(y_pred, y_batch, w_batch)
        loss.backward()
        optimizer.step()

        # Detach and convert to Python float to avoid retaining computation graph
        batch_loss = loss.item()
        batch_losses.append(batch_loss)
        pbar.set_postfix(batch_loss=f"{batch_loss:.4f}")

    # Compute the mean of all individual batch losses
    mean_batch_loss = float(np.mean(batch_losses))
    return mean_batch_loss


def evaluate(model, dataloader, criterion, device):
    """
    Evaluates the model across batches in validation mode and returns the mean loss.
    """
    model.eval()
    val_losses = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Validation Batch", leave=False)
        # FIX 1: Unpack 3 items (input, target, weight_map)
        for X_batch, y_batch, w_batch in pbar:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            w_batch = w_batch.to(device)
            
            y_pred = model(X_batch)
            
            # FIX 2: Pass all 3 arguments to the weighted_loss criterion
            loss = criterion(y_pred, y_batch, w_batch)
            val_losses.append(loss.item())

    return float(np.mean(val_losses))


def train_with_early_stopping(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    max_epochs=30,
    patience=5,
    min_delta=1e-4,
    device="cpu" # ^ fallback
):
    """
    Trains with early stopping and returns the best model along with
    the mean train and validation loss for each epoch.
    """
    best_val_loss = float("inf")
    best_weights = None
    patience_counter = 0

    mean_train_losses = []
    mean_val_losses = []

    epoch_pbar = tqdm(range(1, max_epochs + 1), desc="Epochs")
    for epoch in epoch_pbar:
        # Calculate mean loss of the batches for this epoch
        train_mean_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_mean_loss = evaluate(model, val_loader, criterion, device)

        mean_train_losses.append(train_mean_loss)
        mean_val_losses.append(val_mean_loss)

        # Early stopping check using min_delta tolerance
        if (best_val_loss - val_mean_loss) > min_delta:
            best_val_loss = val_mean_loss
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        epoch_pbar.set_postfix(
            mean_train_loss=f"{train_mean_loss:.4f}",
            mean_val_loss=f"{val_mean_loss:.4f}",
            patience=f"{patience_counter}/{patience}"
        )

        if patience_counter >= patience:
            print(f"\n[Early Stopping] Triggered at epoch {epoch}. Lowest Val Loss: {best_val_loss:.4f}")
            break

    # Restore optimal model state
    if best_weights is not None:
        model.load_state_dict(best_weights)

    return model, mean_train_losses, mean_val_losses

In [ ]:
import matplotlib.pyplot as plt

def plot_diagnostic_curves(train_sizes, train_errors, val_errors):
    """
    Plots the training and validation errors against the size of the training dataset.
    This generates the classic bias-variance learning curve.
    """
    plt.figure(figsize=(10, 6))
    
    # Plot training and validation loss with distinct markers
    plt.plot(train_sizes, train_errors, label='Training Loss', color='blue', marker='o', linestyle='-', linewidth=2, markersize=8)
    plt.plot(train_sizes, val_errors, label='Validation Loss', color='orange', marker='s', linestyle='-', linewidth=2, markersize=8)
    
    # Title and Labels
    plt.title('Learning Curve: Bias-Variance Analysis', fontsize=14, fontweight='bold')
    plt.xlabel('Number of Training Movies (Dataset Size)', fontsize=12)
    plt.ylabel('Mean Weighted Validation/Training Loss', fontsize=12)
    
    # Force X-axis ticks to match your exact dataset increments
    plt.xticks(train_sizes)
    
    # Add grid and legend for readability
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=12, loc='upper right')
    
    plt.tight_layout()
    plt.show()

In [ ]:
import gc
import json
from pathlib import Path
from torch.utils.data import DataLoader
from torch import optim
def main():
    
    data_dir = Path("/home/upmanyu/ml-workspace/Biohub/data/biohub-cell-tracking-during-development/train")

    zarr_files = sorted(list(data_dir.glob("*.zarr")))
    split_idx = int(len(zarr_files) * 0.8)
    train_movies = zarr_files[:split_idx]
    val_movies = zarr_files[split_idx:]

    train_sizes = [20, 40, 80, 120, 160]
    train_errors = []
    val_errors = []

    val_dataset = CellDataset(
                data_dir=data_dir,
                movies=val_movies,
                fpm=10 # ! just a place holder
            )
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=4, pin_memory= True, persistent_workers=True)
    
    for n_movies in train_sizes:

        selected_train_movies = train_movies[:n_movies]

        train_dataset = CellDataset(
            data_dir=data_dir,
            movies=selected_train_movies,
            fpm= 20 # ! just a place holder
        )

        train_loader = train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=4, pin_memory= True, persistent_workers=True)

        model = UNet3D().to(device)

        optimizer = optim.SGD(
            model.parameters(), 
            lr=1e-2  # Note: SGD typically requires a higher learning rate than AdamW (e.g., 1e-2 or 1e-1)
        )

        model, mean_train_losses, mean_val_losses = train_with_early_stopping( # ^ has to be developed should return train and val error best val & train error
            model,
            train_loader,
            val_loader,
            weighted_loss,
            optimizer,
            max_epochs=30,
            patience=5,
            min_delta=1e-4,
            device=device
            
        )
        # calculate the mean of all the train and val loss fucntion and then store in a list and then used it in plotting
        train_errors.append(np.mean(mean_train_losses))
        val_errors.append(np.mean(mean_val_losses))
        del train_loader
        gc.collect()
# ... (end of your loop)

    # Bundle all experiment data
    result = {
    "train_sizes": train_sizes,
    "train_error": train_errors,
    "val_error": val_errors
    }
    weights_dir = Path("/home/upmanyu/ml-workspace/Biohub/notebooks/weights")
    # Save to disk for future plotting or analysis
    output_file = weights_dir / "bias_variance_results.json"
    with open(output_file, "w") as f:
        json.dump(result, f, indent=4)

    print(f"Experiment results saved successfully to {output_file}")
    plot_diagnostic_curves(
        train_sizes=train_sizes,
        train_errors=train_errors,
        val_errors=val_errors
    )

In [ ]:
if __name__ == "__main__":
    main()